# Modular Neuro-Symbolic News Pipeline

This notebook demonstrates a small, understandable slice of the proposed system:

```text
January 2023 news from Hugging Face
→ chronological DeepSeek article assessment
→ temporal event knowledge graph
→ stable fuzzy predicates
→ Logic Tensor Network reasoning
```

The LLM does **not** predict the market regime and does **not** invent final fuzzy
predicate values. It performs semantic tasks that are difficult to express with
fixed Python rules:

- identify the event discussed by an article;
- match it to an existing event;
- determine whether the article introduces, confirms, escalates, contradicts,
  repeats, de-escalates, or resolves that event;
- assign interpretable ordinal attributes using a fixed rubric;
- identify stable economic transmission channels.

Python then:

- validates the structured response;
- updates the temporal event graph;
- maps ordinal attributes into documented fuzzy truth values;
- aggregates article and event evidence into daily predicates;
- evaluates transparent fuzzy logical rules.

## Important scope

The notebook downloads the complete January 2023 partition but defaults to a
small sample for one ticker. Increase `MAX_ARTICLES` only after inspecting the
first outputs.

The notebook checkpoints raw model responses and every event-state change. This
is essential for auditing what the model knew at each point in time.

## 1. University-container setup

Run this installation cell once in the university GPU container. The model is
loaded in BF16 without quantisation. `HF_TOKEN` is loaded from `.env` and is
never printed.

In [44]:
# Uncomment in a fresh university container:
# %pip install -U transformers accelerate huggingface_hub python-dotenv \
#     pandas pyarrow pydantic LTNtorch

In [45]:
from __future__ import annotations

import hashlib
import json
import os
import re
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import hf_hub_download
from pydantic import BaseModel, Field, ValidationError

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 50)

# Search from the current working directory. In the local dissertation workspace,
# also try the sibling Sentiment project without exposing any secret values.

PROJECT_ROOT = Path("/home/jovyan/Stock-Sentiment-Prediction")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

dotenv_path = PROJECT_ROOT / "env.txt"
if not dotenv_path.exists():
    candidates = [
        parent / "Sentiment project" / "env.txt"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ]
    dotenv_path = next((path for path in candidates if path.exists()), None)

if dotenv_path and os.path.exists(dotenv_path):
    load_dotenv(dotenv_path=str(dotenv_path))

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Add it to your env.txt file before continuing."

DATASET_REPO = "cookekieran/alphavantage-market-news"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MONTH = "2023-01"
TICKER = "AAPL"
MIN_TICKER_RELEVANCE = 0.20
MAX_ARTICLES = 15

# Keep True while learning the pipeline. It creates deterministic placeholder
# assessments so every non-model stage can be inspected cheaply.
# Set False in the GPU container to run DeepSeek.
USE_MOCK_LLM = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modular_news_demo" / f"{TICKER}_{MONTH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "dataset": DATASET_REPO,
    "model": MODEL_ID,
    "month": MONTH,
    "ticker": TICKER,
    "max_articles": MAX_ARTICLES,
    "use_mock_llm": USE_MOCK_LLM,
    "output_dir": str(OUTPUT_DIR),
})

{'dataset': 'cookekieran/alphavantage-market-news', 'model': 'Qwen/Qwen2.5-7B-Instruct', 'month': '2023-01', 'ticker': 'AAPL', 'max_articles': 15, 'use_mock_llm': False, 'output_dir': '/home/jovyan/Stock-Sentiment-Prediction/outputs/modular_news_demo/AAPL_2023-01'}


## 2. Download and join the January news partitions

The dataset stores:

- `articles`: title, summary, source, timestamp, and stable `article_uid`;
- `tickers`: ticker relevance and Alpha Vantage ticker sentiment;
- `topics`: topic relevance.

We filter through the ticker-link table rather than relying on
`requested_entity`, because an article requested for one entity may still be
relevant to another.

In [46]:
def download_partition(folder: str) -> Path:
    return Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=f"{folder}/{MONTH}.parquet",
            token=HF_TOKEN,
        )
    )


articles = pd.read_parquet(download_partition("articles"))
tickers = pd.read_parquet(download_partition("tickers"))
topics = pd.read_parquet(download_partition("topics"))

print("articles:", articles.shape)
print("tickers:", tickers.shape)
print("topics:", topics.shape)
display(articles.head(2))

articles: (2344, 11)
tickers: (4220, 7)
topics: (6254, 5)


,article_id,article_uid,time_published,overall_sentiment_score,overall_sentiment_label,title,summary,source,url,requested_entity,month
0,0,cc943125f2c299c334e9484c073748f4,2023-01-02 04:52:00,0.269256,Somewhat-Bullish,What is TSMC?,"Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies...",Tech Monitor,https://www.techmonitor.ai/what-is/what-is-tsmc/,AAPL,2023-01
1,1,58d72b86288d4fffa86b70eac0d65e89,2023-01-03 03:59:02,-0.175983,Somewhat-Bearish,Apple’s market value drops below $2 trillion,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",Al Jazeera,https://www.aljazeera.com/economy/2023/1/3/apples-market-value-drops-below-2-trillion,MSFT,2023-01


In [47]:
ticker_links = (
    tickers.loc[
        (tickers["ticker"] == TICKER)
        & (tickers["relevance_score"] >= MIN_TICKER_RELEVANCE)
    ]
    .rename(
        columns={
            "relevance_score": "ticker_relevance",
            "ticker_sentiment_score": "av_ticker_sentiment_score",
            "ticker_sentiment_label": "av_ticker_sentiment_label",
        }
    )
    [
        [
            "article_uid",
            "ticker",
            "ticker_relevance",
            "av_ticker_sentiment_score",
            "av_ticker_sentiment_label",
        ]
    ]
)

topic_lists = (
    topics.sort_values(["article_uid", "relevance_score"], ascending=[True, False])
    .groupby("article_uid")
    .apply(
        lambda frame: [
            {"topic": row.topic, "relevance": float(row.relevance_score)}
            for row in frame.itertuples()
        ],
        include_groups=False,
    )
    .rename("topics")
    .reset_index()
)

article_stream = (
    articles.merge(ticker_links, on="article_uid", how="inner", validate="one_to_one")
    .merge(topic_lists, on="article_uid", how="left", validate="one_to_one")
    .drop_duplicates("article_uid")
    .sort_values(["time_published", "article_uid"])
    .head(MAX_ARTICLES)
    .reset_index(drop=True)
)
article_stream["topics"] = article_stream["topics"].apply(
    lambda value: value if isinstance(value, list) else []
)

print(f"Selected {len(article_stream)} chronological {TICKER} articles.")
display(
    article_stream[
        [
            "time_published",
            "title",
            "source",
            "ticker_relevance",
            "av_ticker_sentiment_label",
        ]
    ]
)

Selected 15 chronological AAPL articles.


,time_published,title,source,ticker_relevance,av_ticker_sentiment_label
0,2023-01-02 04:52:00,What is TSMC?,Tech Monitor,0.718922,Somewhat-Bullish
1,2023-01-03 03:59:02,Apple’s market value drops below $2 trillion,Al Jazeera,1.000000,Bearish
2,2023-01-03 04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Motley Fool,0.732629,Somewhat-Bullish
3,2023-01-03 04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Globe and Mail,0.749384,Somewhat-Bullish
4,2023-01-03 06:49:00,Foxconn to use Nvidia chips to build self-driving vehicle platforms,Reuters,0.622889,Neutral
5,2023-01-03 12:05:00,Tuesday's ETF with Unusual Volume: IWL,Nasdaq,0.784926,Somewhat-Bearish
6,2023-01-03 17:38:38,Think you have it bad?,The Reformed Broker,0.714321,Somewhat-Bearish
7,2023-01-04 09:26:00,Salesforce to lay off 10% of workforce to cut costs amid economic downturn,Fox Business,0.584422,Somewhat-Bearish
8,2023-01-04 11:52:00,Apple to sign Luxshare for iPhone production in China - FT,Reuters,1.000000,Somewhat-Bullish
9,2023-01-05 03:59:02,Apple brings AI narration to audiobooks,SiliconANGLE,0.981440,Bullish


## 3. Define the LLM's structured output

The LLM returns **ordinal event attributes**, not unexplained decimal scores.
Every category has a stable definition in the prompt.

Event types and transmission channels are intentionally broader than individual
historical events. For example, an article about a specific conflict may create
an event called `Taiwan semiconductor conflict risk`, while its reusable
attributes include:

```text
event_type = geopolitical_conflict
transmission_channels = [supply_chain, operational_disruption]
```

Later, those attributes ground stable predicates such as
`GeopoliticalRisk(AAPL, date)` and `SupplyChainRisk(AAPL, date)`.

In [48]:
Level = Literal["none", "low", "medium", "high", "very_high"]
Direction = Literal["risk_on", "risk_off", "mixed", "neutral"]
Scope = Literal["ticker", "sector", "macro", "broad_market", "irrelevant"]
EventAction = Literal[
    "new",
    "confirm",
    "escalate",
    "deescalate",
    "contradict",
    "repeat",
    "resolve",
    "irrelevant",
]
EventType = Literal[
    "earnings_outlook",
    "product_or_innovation",
    "demand_change",
    "supply_chain",
    "regulatory_or_legal",
    "geopolitical_conflict",
    "management_or_strategy",
    "financing_or_balance_sheet",
    "competitive_position",
    "operational_disruption",
    "macroeconomic",
    "market_sentiment",
    "other",
]
TransmissionChannel = Literal[
    "revenue_growth",
    "demand_weakening",
    "margin_pressure",
    "input_cost_pressure",
    "financing_conditions",
    "supply_chain",
    "operational_disruption",
    "regulatory_restriction",
    "competitive_position",
    "market_uncertainty",
    "none",
]


class ArticleEventAssessment(BaseModel):
    article_uid: str
    ticker: str
    relevant: bool
    matched_event_id: str | None = None
    event_action: EventAction
    event_name: str
    event_type: EventType
    event_summary: str
    direction: Direction
    scope: Scope
    transmission_channels: list[TransmissionChannel]
    materiality: Level
    novelty: Level
    confidence: Level
    evidence_summary: str


LEVEL_VALUE = {
    "none": 0.0,
    "low": 0.25,
    "medium": 0.50,
    "high": 0.75,
    "very_high": 1.0,
}


def model_schema(model_class: type[BaseModel]) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_json_schema"):
        return model_class.model_json_schema()
    return model_class.schema()


def model_dump_compat(model: BaseModel) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model, "model_dump"):
        return model.model_dump()
    return model.dict()


def model_validate_json_compat(model_class: type[BaseModel], payload: str) -> BaseModel:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_validate_json"):
        return model_class.model_validate_json(payload)
    return model_class.parse_raw(payload)

## 4. Temporal event-memory tables

The LLM has no persistent memory between calls. Before each article, Python
supplies a compact context packet containing:

- candidate active events created by earlier articles;
- recent article assessments;
- the current article.

After validation, Python updates three auditable temporal tables:

1. `events`: current event records;
2. `event_state_history`: every historical state change;
3. `event_evidence`: which article caused each change.

Because articles are processed chronologically, a historical call cannot see a
future event update.

In [49]:
def stable_event_id(ticker: str, event_type: str, event_name: str) -> str:
    digest = hashlib.sha1(event_name.lower().encode("utf-8")).hexdigest()[:10]
    return f"{ticker.lower()}__{event_type}__{digest}"


class TemporalEventMemory:
    def __init__(self):
        self.events: dict[str, dict] = {}
        self.state_history: list[dict] = []
        self.evidence: list[dict] = []
        self.recent_assessments = deque(maxlen=6)

    def candidate_events(self, ticker: str, limit: int = 8) -> list[dict]:
        active = [
            event
            for event in self.events.values()
            if event["status"] != "resolved"
            and (event["ticker"] == ticker or event["scope"] in {"sector", "macro", "broad_market"})
        ]
        active.sort(key=lambda event: event["last_updated"], reverse=True)
        return active[:limit]

    def context_packet(self, article: pd.Series) -> dict:
        return {
            "processing_timestamp": pd.Timestamp(article.time_published).isoformat(),
            "ticker": article.ticker,
            "article": {
                "article_uid": article.article_uid,
                "published_at": pd.Timestamp(article.time_published).isoformat(),
                "title": article.title,
                "summary": article.summary,
                "source": article.source,
                "ticker_relevance": float(article.ticker_relevance),
                "topics": article.topics,
            },
            "candidate_active_events": self.candidate_events(article.ticker),
            "recent_assessments": list(self.recent_assessments),
        }

    def apply(self, assessment: ArticleEventAssessment, observed_at: pd.Timestamp) -> str | None:
        if not assessment.relevant or assessment.event_action == "irrelevant":
            self.recent_assessments.append(model_dump_compat(assessment))
            return None

        event_id = assessment.matched_event_id
        if event_id not in self.events:
            event_id = stable_event_id(
                assessment.ticker,
                assessment.event_type,
                assessment.event_name,
            )

        previous = self.events.get(event_id, {})
        status = {
            "new": "unresolved",
            "confirm": previous.get("status", "unresolved"),
            "repeat": previous.get("status", "unresolved"),
            "escalate": "escalating",
            "deescalate": "deescalating",
            "contradict": "contested",
            "resolve": "resolved",
        }[assessment.event_action]

        event = {
            "event_id": event_id,
            "ticker": assessment.ticker,
            "event_name": assessment.event_name,
            "event_type": assessment.event_type,
            "summary": assessment.event_summary,
            "scope": assessment.scope,
            "direction": assessment.direction,
            "transmission_channels": assessment.transmission_channels,
            "materiality": assessment.materiality,
            "confidence": assessment.confidence,
            "status": status,
            "first_seen": previous.get("first_seen", observed_at.isoformat()),
            "last_updated": observed_at.isoformat(),
        }
        self.events[event_id] = event
        self.state_history.append(
            {
                **event,
                "valid_from": observed_at.isoformat(),
                "event_action": assessment.event_action,
            }
        )
        self.evidence.append(
            {
                "event_id": event_id,
                "article_uid": assessment.article_uid,
                "observed_at": observed_at.isoformat(),
                "event_action": assessment.event_action,
                "evidence_summary": assessment.evidence_summary,
            }
        )
        recent = model_dump_compat(assessment)
        recent["resolved_event_id"] = event_id
        self.recent_assessments.append(recent)
        return event_id

    def tables(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        return (
            pd.DataFrame(self.events.values()),
            pd.DataFrame(self.state_history),
            pd.DataFrame(self.evidence),
        )

In [50]:
# Inspect exactly what the first model call can see.
preview_memory = TemporalEventMemory()
first_context_packet = preview_memory.context_packet(article_stream.iloc[0])
print(json.dumps(first_context_packet, indent=2, default=str)[:5000])

{
  "processing_timestamp": "2023-01-02T04:52:00",
  "ticker": "AAPL",
  "article": {
    "article_uid": "cc943125f2c299c334e9484c073748f4",
    "published_at": "2023-01-02T04:52:00",
    "title": "What is TSMC?",
    "summary": "Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies like Apple, Amazon, and Google. It focuses solely on manufacturing for third parties, unlike competitors, and boasts significant market dominance and advanced process technology. The article also discusses the potential impact of a Chinese invasion of Taiwan on TSMC and its global operations, as well as the company's expansion plans in the US.",
    "source": "Tech Monitor",
    "ticker_relevance": 0.718922,
    "topics": [
      {
        "topic": "technology",
        "relevance": 0.943473
      },
      {
        "topic": "manufacturing",
        "relevance": 0.817952
      },
      {
        "topic": "economy_fiscal",
   

## 5. DeepSeek prompt and structured-response parser

DeepSeek receives all instructions in the user prompt. The prompt explicitly
prohibits regime prediction and asks for a final JSON object between markers.

The raw response is retained before parsing. This allows malformed outputs and
model reasoning to be audited later.

In [62]:
RUBRIC = """
Use these ordinal rubrics consistently:

MATERIALITY
- none: no plausible financial consequence
- low: minor commentary or weak operational relevance
- medium: plausible but limited impact on revenue, costs, operations, regulation, or strategy
- high: meaningful impact likely to affect company fundamentals or valuation
- very_high: potentially transformative or severely disruptive impact

NOVELTY
- none: irrelevant or no information
- low: repetition of already-known information
- medium: useful confirmation or modest new detail
- high: important new development
- very_high: unexpected development that substantially changes the known event

CONFIDENCE
- none: unusable evidence
- low: speculation or unclear sourcing
- medium: plausible reporting with uncertainty
- high: well-supported reporting
- very_high: direct authoritative confirmation
"""


def build_prompt(context_packet: dict) -> str:
    schema = model_schema(ArticleEventAssessment)
    return f"""
You are an event analyst for a modular neuro-symbolic financial-news system.

Analyse ONLY the supplied article using the supplied prior context. Do not
predict a market regime, stock return, or price direction. Your task is to
identify or update an event and describe its reusable economic attributes.

Important temporal rule:
- Candidate events and recent assessments are the complete allowed historical
  context.
- Do not assume future knowledge.
- Use matched_event_id only when the article genuinely refers to one of the
  supplied candidate events.
- Repeated reporting should be marked repeat, not new.


You must be extremely strict about declaring an article as "relevant" (relevant: true) to the target ticker.

1. RELEVANCE REQUIREMENT:
   An article is relevant ONLY if it describes a development that directly and causally impacts ticker's specific business operations, revenue channels, cost structure, competitive positioning, or regulatory landscape.

2. WHEN TO CLASSIFY AS IRRELEVANT (relevant: false):
   - Passive Mentions: The article is primarily about another company, market trend, or industry news, and only mentions ticker in passing or as a comparison (e.g., "Company X is competing with ticker").
   - Broad Sector/Macro Noise: The article discusses general macroeconomic trends, broad market indices, or wide sector-level updates (e.g., "Tech stocks rally", "Federal Reserve interest rate changes") WITHOUT specifically linking it to a concrete, direct operational change for ticker.
   - Competitor Action without Direct Reaction: A competitor launches a new product or suffers an outage, but there is no specific, direct operational outcome or structural shift described for target ticker's business.

EVENT MATCHING RULES

Match an article to an existing event only when it discusses the same real-world
development, actors, subject, and causal mechanism.

Shared ticker relevance or transmission channels are not sufficient for a match.

For example:
- An article about Qualcomm launching a competing product must not match an
  existing geopolitical supply-chain event.
- An article mentioning supply chains must not match every existing
  supply-chain-related event.

Use matched_event_id = null and event_action = "new" when the article describes
a distinct development.

Use event_action = "confirm" only when the article provides supporting evidence
that the existing event remains true or has occurred.

If the article does not discuss the existing event, do not use "confirm",
"repeat", "escalate", "deescalate", "contradict", or "resolve".

DEFAULT DECISION: CREATE A NEW EVENT.

Only match an existing event when the article clearly discusses the same
real-world development.

Most articles will NOT match an existing event.

An existing event may be matched only when ALL are true:

1. The core real-world development is the same.
2. The main actors or entities are substantially the same.
3. The causal mechanism is the same.
4. The article explicitly discusses or directly updates the existing event.

The following are NOT sufficient for matching:

- Same ticker
- Same direction
- Same sector
- Same transmission channel
- Both events affecting supply chains
- Both events being financially negative

{RUBRIC}

Return one final JSON object conforming exactly to this JSON schema:
{json.dumps(schema, indent=2)}

Place the final JSON between:
<event_json>
...
</event_json>

Context packet:
{json.dumps(context_packet, indent=2, default=str)}
""".strip()


def normalise_enum(value: object) -> str:
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")


def sanitise_assessment_payload(data: dict, context_packet: dict) -> dict:
    """Repair common, unambiguous formatting errors before validation."""
    allowed_directions = {"risk_on", "risk_off", "mixed", "neutral"}
    allowed_channels = {
        "revenue_growth",
        "demand_weakening",
        "margin_pressure",
        "input_cost_pressure",
        "financing_conditions",
        "supply_chain",
        "operational_disruption",
        "regulatory_restriction",
        "competitive_position",
        "market_uncertainty",
        "none",
    }
    event_type_to_channel = {
        "supply_chain": "supply_chain",
        "operational_disruption": "operational_disruption",
        "regulatory_or_legal": "regulatory_restriction",
        "competitive_position": "competitive_position",
        "geopolitical_conflict": "market_uncertainty",
        "macroeconomic": "market_uncertainty",
        "market_sentiment": "market_uncertainty",
    }

    repaired = dict(data)
    repaired["article_uid"] = context_packet["article"]["article_uid"]
    repaired["ticker"] = context_packet["ticker"]
    repaired["direction"] = normalise_enum(repaired.get("direction", "neutral"))
    if repaired["direction"] == "none":
        repaired["direction"] = "neutral"
    if repaired["direction"] not in allowed_directions:
        repaired["direction"] = "neutral"

    repaired["event_type"] = normalise_enum(repaired.get("event_type", "other"))
    channels = repaired.get("transmission_channels", ["none"])
    if not isinstance(channels, list):
        channels = [channels]
    channels = [normalise_enum(channel) for channel in channels]
    channels = [
        channel if channel in allowed_channels else event_type_to_channel.get(channel)
        for channel in channels
    ]
    repaired["transmission_channels"] = list(
        dict.fromkeys(channel for channel in channels if channel)
    ) or ["none"]

    for field in ("materiality", "novelty", "confidence"):
        repaired[field] = normalise_enum(repaired.get(field, "none"))
        if repaired[field] not in LEVEL_VALUE:
            repaired[field] = "none"
    for field in ("event_action", "scope"):
        repaired[field] = normalise_enum(repaired.get(field, "irrelevant"))
    return repaired


def extract_event_json(raw_text: str, context_packet: dict) -> ArticleEventAssessment:
    match = re.search(r"<event_json>\s*(\{.*?\})\s*</event_json>", raw_text, re.S)
    if not match:
        # Fallback: use the final fenced JSON block.
        blocks = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", raw_text, re.S)
        if not blocks:
            raise ValueError("No event JSON object found in the model response")
        payload = blocks[-1]
    else:
        payload = match.group(1)
    data = json.loads(payload)
    repaired = sanitise_assessment_payload(data, context_packet)
    return model_validate_json_compat(
        ArticleEventAssessment,
        json.dumps(repaired),
    )


print(build_prompt(first_context_packet)[:5000])

You are an event analyst for a modular neuro-symbolic financial-news system.

Analyse ONLY the supplied article using the supplied prior context. Do not
predict a market regime, stock return, or price direction. Your task is to
identify or update an event and describe its reusable economic attributes.

Important temporal rule:
- Candidate events and recent assessments are the complete allowed historical
  context.
- Do not assume future knowledge.
- Use matched_event_id only when the article genuinely refers to one of the
  supplied candidate events.
- Repeated reporting should be marked repeat, not new.


You must be extremely strict about declaring an article as "relevant" (relevant: true) to the target ticker.

1. RELEVANCE REQUIREMENT:
   An article is relevant ONLY if it describes a development that directly and causally impacts ticker's specific business operations, revenue channels, cost structure, competitive positioning, or regulatory landscape.

2. WHEN TO CLASSIFY AS IRRELEV

In [52]:
if not USE_MOCK_LLM:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
    )
    model.eval()


def call_deepseek(context_packet: dict) -> tuple[ArticleEventAssessment, str]:
    prompt = build_prompt(context_packet)
    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(model.device)
        for key, value in encoded.items()
    }

    input_length = encoded["input_ids"].shape[-1]

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=1400,
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_text = tokenizer.decode(
        generated[0, input_length:],
        skip_special_tokens=True,
    )

    return extract_event_json(raw_text, context_packet), raw_text

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 6. Cheap mock mode for understanding the pipeline

Mock mode is deliberately simplistic and must **not** be used as research
output. It lets you inspect event tables, predicate grounding, and LTN reasoning
before spending GPU time.

Set `USE_MOCK_LLM = False` to replace it with DeepSeek.

In [60]:
def mock_assessment(article: pd.Series) -> ArticleEventAssessment:
    score = float(article.av_ticker_sentiment_score)
    direction = "risk_on" if score > 0.15 else "risk_off" if score < -0.15 else "neutral"
    magnitude = abs(score)
    materiality = "high" if magnitude > 0.35 else "medium" if magnitude > 0.15 else "low"
    title_lower = article.title.lower()

    if any(term in title_lower for term in ["supply", "chip", "manufactur"]):
        event_type = "supply_chain"
        channels = ["supply_chain"]
    elif any(term in title_lower for term in ["demand", "sales", "consumer"]):
        event_type = "demand_change"
        channels = ["demand_weakening"] if direction == "risk_off" else ["revenue_growth"]
    elif any(term in title_lower for term in ["market", "stock", "value"]):
        event_type = "market_sentiment"
        channels = ["market_uncertainty"]
    else:
        event_type = "other"
        channels = ["none"]

    return ArticleEventAssessment(
        article_uid=article.article_uid,
        ticker=article.ticker,
        relevant=True,
        matched_event_id=None,
        event_action="new",
        event_name=article.title[:100],
        event_type=event_type,
        event_summary=article.summary[:300],
        direction=direction,
        scope="ticker",
        transmission_channels=channels,
        materiality=materiality,
        novelty="medium",
        confidence="medium",
        evidence_summary="Mock assessment based on existing dataset metadata.",
    )

## 7. Process articles chronologically and build the graph

Each call receives only events and assessments created before the current
article timestamp. Checkpoints are written after every article.

In [61]:
memory = TemporalEventMemory()
assessment_records = []
raw_response_records = []
context_records = []

for index, article in article_stream.iterrows():
    context_packet = memory.context_packet(article)

    try:
        if USE_MOCK_LLM:
            assessment = mock_assessment(article)
            raw_text = "MOCK MODE"
        else:
            assessment, raw_text = call_deepseek(context_packet)
    except Exception as exc:
        raw_response_records.append(
            {
                "article_uid": article.article_uid,
                "time_published": pd.Timestamp(article.time_published).isoformat(),
                "raw_response": locals().get("raw_text", ""),
                "error": f"{type(exc).__name__}: {exc}",
            }
        )
        pd.DataFrame(raw_response_records).to_json(
            OUTPUT_DIR / "raw_model_responses.jsonl",
            orient="records",
            lines=True,
        )
        print(f"Skipped {article.article_uid}: {type(exc).__name__}: {exc}")
        continue

    event_id = memory.apply(assessment, pd.Timestamp(article.time_published))
    assessment_record = {
        **model_dump_compat(assessment),
        "resolved_event_id": event_id,
        "time_published": pd.Timestamp(article.time_published).isoformat(),
    }
    assessment_records.append(assessment_record)
    context_records.append(context_packet)
    raw_response_records.append(
        {
            "article_uid": article.article_uid,
            "time_published": pd.Timestamp(article.time_published).isoformat(),
            "raw_response": raw_text,
        }
    )

    # Auditable checkpoints. A production runner can resume by replaying these
    # validated assessments into an empty TemporalEventMemory instance.
    pd.DataFrame(assessment_records).to_json(
        OUTPUT_DIR / "article_assessments.jsonl",
        orient="records",
        lines=True,
    )
    pd.DataFrame(raw_response_records).to_json(
        OUTPUT_DIR / "raw_model_responses.jsonl",
        orient="records",
        lines=True,
    )
    pd.DataFrame(context_records).to_json(
        OUTPUT_DIR / "context_packets.jsonl",
        orient="records",
        lines=True,
        default_handler=str,
    )
    events_df, states_df, evidence_df = memory.tables()
    events_df.to_parquet(OUTPUT_DIR / "events_current.parquet", index=False)
    states_df.to_parquet(OUTPUT_DIR / "event_state_history.parquet", index=False)
    evidence_df.to_parquet(OUTPUT_DIR / "event_evidence.parquet", index=False)

print(f"Processed {len(assessment_records)} articles.")
assessments_df = pd.DataFrame(assessment_records)
events_df, states_df, evidence_df = memory.tables()
display(assessments_df.head(10))

Skipped cc943125f2c299c334e9484c073748f4: ValidationError: 1 validation error for ArticleEventAssessment
event_name
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
Skipped aa9775754e1566cebaeea81ca19ecca4: ValidationError: 3 validation errors for ArticleEventAssessment
event_name
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
event_type
  Input should be 'earnings_outlook', 'product_or_innovation', 'demand_change', 'supply_chain', 'regulatory_or_legal', 'geopolitical_conflict', 'management_or_strategy', 'financing_or_balance_sheet', 'competitive_position', 'operational_disruption', 'macroeconomic', 'market_sentiment' or 'other' [type=literal_error, input_value='none', input_type=str]
    For further information visit https://errors.pydantic.

,article_uid,ticker,relevant,matched_event_id,event_action,event_name,event_type,event_summary,direction,scope,transmission_channels,materiality,novelty,confidence,evidence_summary,resolved_event_id,time_published
0,58d72b86288d4fffa86b70eac0d65e89,AAPL,True,NaN,new,Market Value Drop,financing_or_balance_sheet,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],medium,high,high,"The article directly reports a significant drop in Apple's market capitalization, which is a key financial metric that can affect invest...",aapl__financing_or_balance_sheet__e30aecc219,2023-01-03T03:59:02
1,eec1c245936a45f5e7e4f9134ac6e54d,AAPL,True,aapl__financing_or_balance_sheet__e30aecc219,confirm,Market Value Drop,financing_or_balance_sheet,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],medium,medium,high,"The article confirms the significant drop in Apple's market capitalization, which is consistent with the existing event and provides add...",aapl__financing_or_balance_sheet__e30aecc219,2023-01-03T12:05:00
2,1f45b4cbc18d012cd1fd6cdb1201c7f7,AAPL,False,NaN,irrelevant,Economic Environment Discussion,macroeconomic,"The article discusses the challenging economic conditions of 2022, including market downturns and specific impacts on tech startups and ...",neutral,irrelevant,[none],none,none,low,The article primarily focuses on general economic conditions and their impacts on specific industries rather than providing specific dev...,NaN,2023-01-03T17:38:38
3,f1a28840cb9391fc012bce188488f0e4,AAPL,False,NaN,irrelevant,Irrelevant Article on Salesforce Layoffs,other,Salesforce announced it will lay off 10% of its workforce and reduce office space to cut operating costs amid a challenging economic env...,neutral,irrelevant,[none],none,none,low,"The article primarily focuses on Salesforce's cost-cutting measures and does not provide any specific, direct operational changes or imp...",NaN,2023-01-04T09:26:00
4,6217c394b1954da3f19674ff41f5fbe9,AAPL,True,NaN,new,Supply Chain Diversification,supply_chain,Apple is reportedly set to partner with Chinese contract manufacturer Luxshare Precision Industry Co Ltd for the production of premium i...,neutral,ticker,[supply_chain],medium,high,high,"The article directly reports a new strategic partnership between Apple and Luxshare, which is a significant development in Apple's suppl...",aapl__supply_chain__cb63444c7f,2023-01-04T11:52:00
5,21f04789fa15efceab5ae127c932b985,AAPL,True,NaN,new,AI Narration in Audiobooks,product_or_innovation,"Apple Inc. has introduced AI-narrated audiobooks through Apple Books, utilizing text-to-speech technology to make audiobook production m...",neutral,ticker,"[revenue_growth, demand_weakening, margin_pressure, input_cost_pressure]",low,high,high,"The article directly describes a new product innovation from Apple, which introduces AI-narrated audiobooks. This development could have...",aapl__product_or_innovation__3e46f63a42,2023-01-05T03:59:02
6,81aa1489ed05c97cebd528aca2b0d057,AAPL,False,NaN,irrelevant,Irrelevant Article on Qualcomm's New Service,other,"Qualcomm is introducing Snapdragon Satellite service, a new emergency satellite phone service for Android smartphones. This service aims...",neutral,irrelevant,[none],none,none,low,"The article primarily focuses on Qualcomm's new satellite service for Android smartphones, which does not provide any specific, direct o...",NaN,2023-01-05T15:29:00
7,a525cf05a0a5b0536550ca77425ae4ae,AAPL,False,NaN,irrelevant,Nordstrom Board Appointment,other,"Nordstrom, Inc. has appointed Atticus Tysen to its board of directors, bringing extensive engineering and information security expertise...",neutral,irrelevant,[none],none,none,low,"The article primarily focuses on Nordstrom's board appointment and 

In [27]:
errors_df = pd.read_json(
    OUTPUT_DIR / "raw_model_responses.jsonl",
    lines=True,
)

for uid in [
    "a525cf05a0a5b0536550ca77425ae4ae",
    "60d54f8ae87e43840597ca0f7dd2f711",
]:
    response = errors_df.loc[
        errors_df["article_uid"] == uid,
        "raw_response",
    ].iloc[0]

    print(response)

<event_json>
{
  "article_uid": "a525cf05a0a5b0536550ca77425ae4ae",
  "ticker": "AAPL",
  "relevant": false,
  "matched_event_id": null,
  "event_action": "new",
  "event_name": "Board Appointments - Atticus Tysen to Nordstrom Board",
  "event_type": "management_or_strategy",
  "event_summary": "Nordstrom, Inc. has appointed Atticus Tysen to its board of directors, effective January 3, 2023. Tysen brings extensive experience in engineering and information security, currently serving as senior vice president of product development, chief information security, and fraud prevention officer for Intuit.",
  "direction": "neutral",
  "scope": "irrelevant",
  "transmission_channels": [],
  "materiality": "none",
  "novelty": "high",
  "confidence": "high",
  "evidence_summary": "The article reports on a new board appointment at Nordstrom, highlighting the addition of Atticus Tysen with relevant expertise. This information does not pertain to any of the existing events related to Apple."
}
</e

In [ ]:
print("Current events")
display(events_df)

print("Event state history")
display(states_df.head(10))

print("Event evidence")
display(evidence_df.head(10))

## 8. Ground event attributes into stable predicates

The event graph preserves specific facts. Predicates express reusable logical
statements.

Example:

```text
Specific event:
    "Taiwan semiconductor conflict risk"

Stable predicates:
    GeopoliticalRisk(AAPL, date)
    SupplyChainRisk(AAPL, date)
    UnresolvedMaterialRisk(AAPL, date)
```

The grounding functions below are documented and deterministic. Therefore,
`NewsRiskOff(AAPL, date) = 0.72` is not an arbitrary dictionary value: it is the
truth degree returned by a defined function applied to validated evidence.

In [25]:
def fuzzy_or(values) -> float:
    result = 0.0
    for value in values:
        value = max(0.0, min(1.0, float(value)))
        result = result + value - result * value
    return float(result)


def article_truths(row: pd.Series) -> dict[str, float]:
    materiality = LEVEL_VALUE[row.materiality]
    novelty = LEVEL_VALUE[row.novelty]
    confidence = LEVEL_VALUE[row.confidence]
    relevance = float(
        article_stream.set_index("article_uid").loc[row.article_uid, "ticker_relevance"]
    )
    strength = materiality * confidence * relevance

    return {
        "news_risk_on": strength if row.direction == "risk_on" else 0.0,
        "news_risk_off": strength if row.direction == "risk_off" else 0.0,
        "news_material": materiality,
        "news_novel": novelty,
        "ticker_specific_news": relevance if row.scope == "ticker" else 0.0,
        "new_material_event": strength * novelty if row.event_action == "new" else 0.0,
        "existing_event_escalated": strength if row.event_action == "escalate" else 0.0,
        "previously_known_information": strength * (1.0 - novelty)
        if row.event_action == "repeat"
        else 0.0,
    }


def active_event_truths(states: pd.DataFrame, as_of: pd.Timestamp) -> dict[str, float]:
    if states.empty:
        return {}
    history = states.copy()
    history["valid_from"] = pd.to_datetime(history["valid_from"])
    visible = history.loc[history["valid_from"] <= as_of].sort_values("valid_from")
    current = visible.groupby("event_id", as_index=False).tail(1)
    current = current.loc[current["status"] != "resolved"]

    truths: dict[str, list[float]] = {
        "unresolved_material_risk": [],
        "geopolitical_risk": [],
        "regulatory_risk": [],
        "supply_chain_risk": [],
        "demand_weakening": [],
        "input_cost_pressure": [],
        "operational_disruption": [],
    }

    for event in current.itertuples():
        age_days = max(0, (as_of - pd.Timestamp(event.valid_from)).days)
        decay = 0.5 ** (age_days / 30.0)
        strength = LEVEL_VALUE[event.materiality] * LEVEL_VALUE[event.confidence] * decay

        if event.direction == "risk_off":
            truths["unresolved_material_risk"].append(strength)
        if event.event_type == "geopolitical_conflict":
            truths["geopolitical_risk"].append(strength)
        if event.event_type == "regulatory_or_legal":
            truths["regulatory_risk"].append(strength)
        if "supply_chain" in event.transmission_channels:
            truths["supply_chain_risk"].append(strength)
        if "demand_weakening" in event.transmission_channels:
            truths["demand_weakening"].append(strength)
        if "input_cost_pressure" in event.transmission_channels:
            truths["input_cost_pressure"].append(strength)
        if "operational_disruption" in event.transmission_channels:
            truths["operational_disruption"].append(strength)

    return {name: fuzzy_or(values) for name, values in truths.items()}

In [26]:
assessment_truth_rows = []

for row in assessments_df.itertuples(index=False):
    assessment_truth_rows.append({
        "article_uid": row.article_uid,
        "date": pd.Timestamp(row.time_published).normalize(),
        **article_truths(pd.Series(row._asdict())),
    })

article_truth_df = pd.DataFrame(assessment_truth_rows)

if article_truth_df.empty:
    raise ValueError(
        "No valid article assessments were produced. "
        "Inspect raw_model_responses.jsonl for parsing or validation errors."
    )

required_columns = {"article_uid", "date"}
missing = required_columns - set(article_truth_df.columns)

if missing:
    raise ValueError(
        f"Article truth table is missing columns: {sorted(missing)}"
    )

daily_rows = []

for date, day in article_truth_df.groupby("date"):
    row = {"ticker": TICKER, "date": date}

    predicate_columns = [
        column
        for column in day.columns
        if column not in {"article_uid", "date"}
    ]

    for predicate in predicate_columns:
        row[predicate] = fuzzy_or(day[predicate].tolist())

    row["conflicting_news"] = min(
        row.get("news_risk_on", 0.0),
        row.get("news_risk_off", 0.0),
    )

    row.update(
        active_event_truths(
            states_df,
            pd.Timestamp(date)
            + pd.Timedelta(days=1)
            - pd.Timedelta(seconds=1),
        )
    )

    daily_rows.append(row)

daily_predicates = pd.DataFrame(daily_rows).sort_values("date").reset_index(drop=True)
display(daily_predicates)

,ticker,date,news_risk_on,news_risk_off,news_material,news_novel,ticker_specific_news,new_material_event,existing_event_escalated,previously_known_information,conflicting_news,unresolved_material_risk,geopolitical_risk,regulatory_risk,supply_chain_risk,demand_weakening,input_cost_pressure,operational_disruption
0,AAPL,2023-01-02,0.0,0.000000,0.5000,0.5000,0.000000,0.089865,0.0,0.000000,0.0,0.0000,0.0,0.0,0.250000,0.0,0.0,0.0
1,AAPL,2023-01-03,0.0,0.000000,0.7500,0.4375,0.000000,0.000000,0.0,0.301664,0.0,0.0000,0.0,0.0,0.250000,0.0,0.0,0.0
2,AAPL,2023-01-06,0.0,0.000000,0.4375,0.4375,0.000000,0.020121,0.0,0.046875,0.0,0.0000,0.0,0.0,0.233258,0.0,0.0,0.0
3,AAPL,2023-01-10,0.0,0.370193,0.7500,0.7500,0.913329,0.195590,0.0,0.000000,0.0,0.4375,0.0,0.0,0.212667,0.0,0.0,0.0


In [27]:
errors_df = pd.read_json(
    OUTPUT_DIR / "raw_model_responses.jsonl",
    lines=True,
)

print(errors_df["error"].value_counts(dropna=False))
display(errors_df[["article_uid", "error", "raw_response"]].head())

error
NaN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  7
ValueError: No event JSON object found in the model response                                                                                                                                                                               

,article_uid,error,raw_response
0,cc943125f2c299c334e9484c073748f4,NaN,"Okay, so I need to analyze the provided article using the given context to identify or update an event and describe its reusable economi..."
1,58d72b86288d4fffa86b70eac0d65e89,NaN,"Alright, let's tackle this analysis. I need to process the given article and determine if it refers to any candidate events, update rele..."
2,aa9775754e1566cebaeea81ca19ecca4,"ValidationError: 2 validation errors for ArticleEventAssessment\nevent_type\n Input should be 'earnings_outlook', 'product_or_innovatio...","Alright, let's tackle this analysis. I need to process the given article and determine if it refers to any candidate events, update rele..."
3,e94cd66eaeab77a193199183b1a77cbc,NaN,"Alright, let's break down how I approached this task. The user provided an article and some context about candidate events, and my job w..."
4,cd94864135b4019d7f7e9be93d4d9044,ValueError: No event JSON object found in the model response,"Alright, let's break down how I approached this task. The user provided an article and some context about candidate events, and my job w..."


## 9. LTN reasoning over the grounded predicates

LTN does not magically discover the meaning of predicates. The previous cells
performed their grounding.

Here, LTN fuzzy operators evaluate explicit reusable rules:

```text
NewsRiskOff ∧ NewsMaterial ∧ UnresolvedMaterialRisk → BearSupport
NewsRiskOn ∧ NewsMaterial ∧ ¬UnresolvedMaterialRisk → BullSupport
ConflictingNews ∨ (NewsMaterial ∧ ¬NewsRiskOn ∧ ¬NewsRiskOff) → NeutralSupport
```

This is a transparent inference demonstration. Later, the regime predicates or
rule weights can be made learnable using labelled regime data and LTN
satisfiability as a training objective.

In [ ]:
try:
    import ltn

    And = ltn.fuzzy_ops.AndProd()
    Or = ltn.fuzzy_ops.OrProbSum()
    Not = ltn.fuzzy_ops.NotStandard()
    Implies = ltn.fuzzy_ops.ImpliesReichenbach()
    LTN_BACKEND = "LTNtorch"
except ImportError:
    # Transparent fallback for inspecting the notebook before LTNtorch is installed.
    And = lambda x, y: x * y
    Or = lambda x, y: x + y - x * y
    Not = lambda x: 1.0 - x
    Implies = lambda x, y: 1.0 - x + x * y
    LTN_BACKEND = "transparent PyTorch fallback"

print("Logic backend:", LTN_BACKEND)


def tensor_column(frame: pd.DataFrame, name: str) -> torch.Tensor:
    if name not in frame:
        return torch.zeros(len(frame), dtype=torch.float32)
    return torch.tensor(frame[name].fillna(0.0).to_numpy(), dtype=torch.float32)


risk_off = tensor_column(daily_predicates, "news_risk_off")
risk_on = tensor_column(daily_predicates, "news_risk_on")
material = tensor_column(daily_predicates, "news_material")
unresolved = tensor_column(daily_predicates, "unresolved_material_risk")
conflicting = tensor_column(daily_predicates, "conflicting_news")
escalated = tensor_column(daily_predicates, "existing_event_escalated")
geopolitical = tensor_column(daily_predicates, "geopolitical_risk")
supply_chain = tensor_column(daily_predicates, "supply_chain_risk")

bear_rule_1 = And(And(risk_off, material), unresolved)
bear_rule_2 = And(And(escalated, geopolitical), supply_chain)
bear_support = Or(bear_rule_1, bear_rule_2)

bull_support = And(And(risk_on, material), Not(unresolved))
weak_direction = And(Not(risk_on), Not(risk_off))
neutral_support = Or(conflicting, And(material, weak_direction))

scores = torch.stack([bear_support, neutral_support, bull_support], dim=1)
scores = scores / scores.sum(dim=1, keepdim=True).clamp_min(1e-8)

ltn_results = daily_predicates[["ticker", "date"]].copy()
ltn_results[["bear_support", "neutral_support", "bull_support"]] = scores.detach().numpy()
ltn_results["detected_regime"] = [
    ["bear_drawdown", "sideways_neutral", "bull_rally"][index]
    for index in scores.argmax(dim=1).tolist()
]
display(ltn_results)

In [17]:
# Formula-satisfaction diagnostics illustrate how LTN evaluates implications.
formula_satisfaction = pd.DataFrame(
    {
        "date": daily_predicates["date"],
        "risk_off_material_unresolved_implies_bear": Implies(
            bear_rule_1, scores[:, 0]
        ).detach().numpy(),
        "risk_on_material_implies_bull": Implies(
            And(risk_on, material), scores[:, 2]
        ).detach().numpy(),
        "conflicting_news_implies_neutral": Implies(
            conflicting, scores[:, 1]
        ).detach().numpy(),
    }
)
display(formula_satisfaction)

NameError: name 'daily_predicates' is not defined

## 10. Inspect one prediction from evidence to rule

This final view connects:

```text
source article
→ article assessment
→ event/evidence record
→ daily predicate truth values
→ activated LTN rule
→ regime support
```

In [18]:
if not daily_predicates.empty:
    selected_date = daily_predicates.iloc[-1]["date"]
    print("Selected date:", selected_date)

    print("\nArticles and assessments")
    display(
        assessments_df.loc[
            pd.to_datetime(assessments_df["time_published"]).dt.normalize() == selected_date
        ]
    )

    print("\nVisible event states")
    display(
        states_df.loc[
            pd.to_datetime(states_df["valid_from"])
            <= pd.Timestamp(selected_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        ]
    )

    print("\nGrounded predicates")
    display(daily_predicates.loc[daily_predicates["date"] == selected_date])

    print("\nLTN result")
    display(ltn_results.loc[ltn_results["date"] == selected_date])

NameError: name 'daily_predicates' is not defined

## What to inspect before scaling

Do not process the full dataset yet. First inspect:

1. Does DeepSeek consistently distinguish new events from repeated reporting?
2. Do matched event IDs refer only to supplied candidate events?
3. Are event types and transmission channels stable across similar articles?
4. Do ordinal materiality and novelty assignments follow the rubric?
5. Do event-state histories contain only information available at each timestamp?
6. Are predicate truth values traceable to source evidence?
7. Are the LTN rules economically defensible?

After this works for a small January sample, increase `MAX_ARTICLES`, evaluate
multiple tickers, and replace calendar-day aggregation with an exchange-calendar
function that assigns after-hours and weekend news to the correct trading date.